---

<center> <h1> <b> <span style='color:#292D78'> CREWES ML Meetings </span> </b> </h1> </center>

<center> <h2> <b> <span style='color:#DF7F00'> RAG for Chat Assistant </span> </b> </h2> </center>

---

In this [Jupyter Notebook](https://jupyter.org/install) we will work on creating a database from PDF files to use in the chat assistant.

# **Project Description**

Create the **CREWES Copilot**, a chat assistant, that can answer scientific questions related to the CREWES reports from 2020 using [Retrieval-Augmented Generation](https://research.ibm.com/blog/retrieval-augmented-generation-RAG).

It will require a sequence of processes that starts from mining the data from the [CREWES website](https://www.crewes.org/), and ends by creating the chat assistant powered by `gpt-4o-mini`:

1. Download CREWES reports using the libraries [requests](https://pypi.org/project/requests/) and [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/).
2. Load the PDF files and split them into chunks using [langchain_text_splitters](https://api.python.langchain.com/en/latest/text_splitters_api_reference.html).
3. Embedding the chunks using an [embedding model](https://platform.openai.com/docs/guides/embeddings).
4. Create a [Chroma](https://www.trychroma.com/) database.
5. Receive an user message and embed it.
6. Retrieve the `k` most similar chunks to the user message.
7. Use [gpt-4o-mini](https://openai.com/index/gpt-4o-mini-advancing-cost-efficient-intelligence/) to generate a response based on the retrieved chunks.

# Libraries

In [ ]:
# Uncomment the following code to install the required packages
# !pip install -q langchain==0.1.16 langchain_openai==0.1.3 langchain-experimental==0.0.57 langchain-community==0.0.34 streamlit==1.33.0 pypdf==4.2.0 google-api-python-client pytube==15.0.0 youtube-transcript-api==0.6.2 PyYAML==6.0.1 chromadb==0.5.0 watchdog sqlalchemy==2.0.29 requests beautifulsoup4 urllib3

In [1]:
# Libraries for data mining
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Libraries to create database and test the OpenAI model
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

import yaml
from pprint import pprint

# Downloading CREWES Reports

For the chat assistant, we need to create a database containing information of the CREWES research topics. For the demonstration, let's mine the [CREWES reports from 2020](https://www.crewes.org/Documents/ResearchReports/reports.php?year=2020) and download them.

First, getting the `HTML` code of the target *URL*.

In [2]:
# URL to scrape
url = "https://www.crewes.org/Documents/ResearchReports/reports.php?year=2020" # url to scrape

# If there is no such folder, the script will create one automatically
folder_location = r'./pdf' # folder location

# create folder if it doesn't exist
if not os.path.exists(folder_location):os.mkdir(folder_location)
 
response = requests.get(url) # get the html
soup = BeautifulSoup(response.text, "html.parser") # parse the html

In [3]:
soup


<!DOCTYPE html>

<html lang="en">
<head>
<title>Research Reports</title>
<!-- Required meta tags -->
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<!-- Bootstrap 5 CSS -->
<link crossorigin="anonymous" href="https://cdn.jsdelivr.net/npm/bootstrap@5.0.1/dist/css/bootstrap.min.css" integrity="sha384-+0n0xVW2eSR5OomGNYDnhzAbDsOXxcvSN1TPprVMTNDbiYZCxYbOOl7+AMvyTG2x" rel="stylesheet"/>
<!-- Add icon library -->
<link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.min.css" rel="stylesheet"/>
<!-- Add local css -->
<link href="/common/css/crewes-theme.css" rel="stylesheet" type="text/css"/>
</head>
<body class="bg-light">
<!-- START NAVBAR -->
<header class="navbar navbar-expand-lg navbar-light bg-light bd-navbar fixed-top shadow">
<nav aria-label="Main navigation" class="container-lg flex-wrap">
<!--div class="container-fluid"-->
<!--div class="navbar"-->
<!-- CREWES LOGO -->
<a aria-label="CREWES" class="navb

In [4]:
import re

In [7]:
author = re.findall(r"Innanen", soup.text)
len(author)

40

Locate all the links to a `PDF` file and download them.

In [ ]:
for link in soup.select("a[href$='.pdf']"): # select all the pdf links
    # Name the pdf files using the last portion of each link which are unique in this case
    filename = os.path.join(folder_location,link['href'].split('/')[-1]) # join the folder location and the filename
    with open(filename, 'wb') as f: 
    # open the file and write the pdf
        f.write(requests.get(urljoin(url,link['href'])).content)

# Create Database using Chroma

Importing **YOUR** [OpenAI API Key](https://help.openai.com/en/articles/4936850-where-do-i-find-my-openai-api-key) to the environment.

In [12]:
OPENAI_API_KEY = yaml.safe_load(open('../info.yml'))['openai']

Using `langchain` to load all the *pdf* files to create the database.

In [20]:
loader = PyPDFDirectoryLoader("./pdf")
documents = loader.load() # this can take a few minutes to run

In [21]:
len(documents)

1017

In [22]:
documents[100]

Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2020-12-02T16:35:12-07:00', 'author': 'Ninoska', 'subject': 'CREWES Research Report', 'moddate': '2020-12-02T16:35:12-07:00', 'source': 'pdf\\CRR202002.pdf', 'total_pages': 17, 'page': 4, 'page_label': '5'}, page_content='FWI time-lapse \n CREWES Research Report — Volume 32 (2020) 5 \nand fine features are loss once the model is smoothed. A full view of our true models is \npresented in Figure 3. \n \nFIG. 2. Profile comparison between the P -wave velocity derived from well-logs and the smoothed \nvelocity adaptation considered in this investigation for modeling a (a) baseline, (b) one year of CO2 \ninjection and (c) five years of CO2 injection. \nTwo profile locations were defined at 40 m and 190 m distanc e from the receiver \narrangement to evaluate the inversion updates in detail at near and far offsets, as indicated \nin Figure 3(b). These locations 

Sppliting all the reports into chunks. The splitter used here is recursive. It looks for specific characters in order to make a split: two new lines (`\n\n`), followed by one new line (`\n`), then a space (`" "`), and finally a character (`""`).

In [23]:
CHUNK_SIZE = 500

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = CHUNK_SIZE, 
    chunk_overlap = 50,
)

docs = text_splitter.split_documents(documents)
print(len(docs))

5012


In [24]:
docs[100]

Document(metadata={'producer': 'Adobe PDF Library 11.0', 'creator': 'Acrobat PDFMaker 11 for Word', 'creationdate': '2020-12-02T13:52:08-07:00', 'author': 'Kevin', 'company': '', 'moddate': '2020-12-02T13:55:47-07:00', 'sourcemodified': 'D:20201202205132', 'title': '', 'source': 'pdf\\CRR202000.pdf', 'total_pages': 86, 'page': 28, 'page_label': '29'}, page_content='inversion approach to real datasets of PP and PS wave s, we obtain reliable results of \nfracture weaknesses that can match well log curve of velocity and anisotropic AVA \ngradient, which may provide the possibility to characterize how fractures distribute and to \nestimate fracture connectivity in hydrocarbon reservoirs. \n \n \nFIG. 1. a) Inversion results  of the normal fracture weakness δ N, and b) Inversion results of the \ntangential fracture weakness δT. The curve is P-wave velocity.')

# Embedding Sentences

Using the OpenAI `text-embedding-3-small` model to convert each chunk from text to vectors with 1536 float numbers.

In [13]:
my_embedding = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    api_key = OPENAI_API_KEY
)

# Create Database

Creating a **Chroma** database with all the chunks and their embeddings.

In [ ]:
vectorstore = Chroma.from_documents(
    docs, 
    persist_directory = "./data/chroma_CREWES.db",
    embedding = my_embedding
)

# Load Database

Loading the database as a retrieval object.

In [25]:
vectorstore = Chroma(
    persist_directory = "./data/chroma_CREWES.db",
    embedding_function = my_embedding
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

retriever.invoke("What is full-waveform inversion?")

[Document(metadata={'page': 0, 'source': 'pdf\\CRR202054.pdf'}, page_content='INTRODUCTION\nFull Waveform Inversion (FWI) is one of the most powerful but also difﬁcult inversion\nalgorithms to apply in real data processing (Schuster, 2017). It can bring up detailed ve-\nlocity information about the subsurface by ﬁtting modeled data to observations. Its power\ncan be easily proven for synthetic data in ideal conditions but in practice, many difﬁculties\nneed to be overcome to reach its goals. Some of these issues are related to the non-linearity'),
 Document(metadata={'page': 13, 'source': 'pdf\\CRR202040.pdf'}, page_content='full waveform inversion, inEncyclopedia of exploration geophysics, Society of Exploration Geophysicists,\nR1–1.\nVirieux, J., and Operto, S., 2009, An overview of full-waveform inversion in exploration geophysics: Geo-\nphysics, 74, No. 6, WCC1–WCC26.\nWomack, J., Cruz, J., Rigdon, H., and Hoover, G., 1990, Encoding techniques for multiple source point\nseismic dat

# Chat Assistant with RAG

Create a system prompt that set the model to generate responses based only on the retrieved chunks from the database.

In [15]:
template = """Answer the question based only on the following context. Do not use any other information:
{context}

In the answers, include equations whenever necessary, as well the sources of the information (references and the pdf file it came from, including the page number. Remove "pdf////" from the file name). Return the answers in markdown format.

Question: {question}
"""

Setting the model.

In [16]:
# Convert the prompt template to a ChatPromptTemplate object
prompt = ChatPromptTemplate.from_template(template)

# Setting the model
model = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0.7,
    api_key = OPENAI_API_KEY
)

Creating the pipeline for requests and responses.

In [17]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [26]:
rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001C30C2593D0>, search_kwargs={'k': 6}),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context. Do not use any other information:\n{context}\n\nIn the answers, include equations whenever necessary, as well the sources of the information (references and the pdf file it came from, including the page number. Remove "pdf////" from the file name). Return the answers in markdown format.\n\nQuestion: {question}\n'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001C30C29EE50>, async_client=<openai.

Testing the assistant:

In [30]:
result = rag_chain.invoke("What are converted waves?")

print(result)

Converted waves refer to seismic waves that have changed from one type of wave to another as they propagate through different media. For example, when a P-wave (primary or compressional wave) encounters a boundary with a different medium, it can convert into an S-wave (secondary or shear wave) and vice versa. This phenomenon typically occurs at interfaces where the physical properties of the materials, such as density and velocity, differ.

In the context provided, while there is no direct mention of the term "converted waves," the discussion around wave propagation involves understanding how waves behave when they encounter different media, which is fundamental to the concept of converted waves. This is illustrated in the synthetic wavefields and their components, where different wave types interact through the propagation medium (CRR202039.pdf, page 4).

For a more detailed understanding, one can refer to the equations governing wave propagation and conversion at material interfaces,

Converted waves refer to seismic waves that have changed from one type of wave to another as they propagate through different media. For example, when a P-wave (primary or compressional wave) encounters a boundary with a different medium, it can convert into an S-wave (secondary or shear wave) and vice versa. This phenomenon typically occurs at interfaces where the physical properties of the materials, such as density and velocity, differ.

In the context provided, while there is no direct mention of the term "converted waves," the discussion around wave propagation involves understanding how waves behave when they encounter different media, which is fundamental to the concept of converted waves. This is illustrated in the synthetic wavefields and their components, where different wave types interact through the propagation medium (CRR202039.pdf, page 4).

For a more detailed understanding, one can refer to the equations governing wave propagation and conversion at material interfaces, although specific equations are not provided in the given context.

Full Waveform Inversion (FWI) is a high-resolution seismic imaging technique that utilizes the entire content of seismic traces to extract physical parameters of the subsurface medium sampled by seismic waves. FWI aims to invert the velocity model by minimizing the difference between observed and simulated seismic data (Tarantola, 1984). It is particularly effective in capturing subsurface properties that conventional velocity analysis may not resolve, making it suitable for applications such as reservoir monitoring (Smithyman et al., 2015; Pan et al., 2017).

Mathematically, FWI is described as a PDE-constrained optimization problem with a nonlinear nonconvex objective function. The nonconvex nature of the problem implies that local minima are unavoidable in the optimization process, especially when the initial model is significantly distant from the true model (Virieux and Operto, 2009; Virieux et al., 2017).

### References
- Tarantola, A. (1984). *Classical time-domain full-waveform inversion*.
- Smithyman et al., (2015). *Applications of FWI for reservoir monitoring*. (CRR202002, page 0)
- Pan et al., (2017). *Applications of FWI for reservoir monitoring*. (CRR202002, page 0)
- Virieux and Operto, (2009). *General review of the FWI problem*. (CRR202038, page 0)
- Virieux et al., (2017). *General review of the FWI problem*. (CRR202038, page 0)

# **Thank you!!!**